In [1]:
#concatenar los resultados de las 3 fuentes de datos
import pandas as pd
import numpy as np
dfdrive = pd.read_parquet('dfdriveready.parquet')
df123 = pd.read_csv('../1.4 parquet_123/123oldloadsready.csv')
dfaws = pd.read_parquet('dfawsready.parquet')

/tmp/ipykernel_2256167/4138636736.py:5: DtypeWarning: Columns (17: Broker_Shipper) have mixed types. Specify dtype option on import or set low_memory=False.
  df123 = pd.read_csv('../1.4 parquet_123/123oldloadsready.csv')


In [2]:
cols = ["Posted",'PickUp']
dfaws[cols] = dfaws[cols].apply(pd.to_datetime, format="mixed")

In [3]:
cols = ["Posted",'PickUp']
df123[cols] = df123[cols].apply(pd.to_datetime, format="mixed")
dic2 = {'Size_ft':'Size'}
df123 = df123.rename(columns=dic2)

In [4]:
cols=['ID', 'Posted', 'PickUp', 'CityOrigin', 'LatOrigin', 'LngOrigin', 'CityDestination',
      'LatDestination', 'LngDestination', 'Distance', 'Source', 'Equip','StateOrigin',
      'HubOrigin', 'StateDestination', 'HubDestination', 'RatePerMile',
        "Weight",
        "Size"]
df123 = df123[cols]
dfaws = dfaws[cols]

In [5]:
df123[['ID','Equip', 'Weight', 'Size']] = df123[['ID','Equip', 'Weight', 'Size']].fillna('').astype(str)
df123['Equip'] = df123['Equip'].to_numpy()
#df123[['Posted', 'PickUp']] = df123[['Posted', 'PickUp']].astype(str).replace('nan', '').replace('', np.nan)
df123[['LatOrigin', 'LatDestination', 'LngOrigin', 'LngDestination']] = df123[['LatOrigin', 'LatDestination', 'LngOrigin', 'LngDestination']].astype(float)

In [6]:
dfaws[['ID','Equip', 'Weight', 'Size']] = dfaws[['ID','Equip', 'Weight', 'Size']].fillna('').astype(str)
dfaws['Equip'] = dfaws['Equip'].to_numpy()
#df123[['Posted', 'PickUp']] = df123[['Posted', 'PickUp']].astype(str).replace('nan', '').replace('', np.nan)
dfaws[['LatOrigin', 'LatDestination', 'LngOrigin', 'LngDestination']] = dfaws[['LatOrigin', 'LatDestination', 'LngOrigin', 'LngDestination']].astype(float)

In [7]:
df_full = pd.concat([df123,dfaws,dfdrive])

In [8]:
df_full.shape

(70631726, 20)

In [9]:
df_dupp = df_full.drop_duplicates(subset=["ID"], keep="last")
df_dupp.shape

(62544534, 20)

In [10]:
from clean_data import remove_duplicates_general
dfgenduplicates = remove_duplicates_general(df_dupp)
dfgenduplicates.shape

['ID', 'Posted', 'PickUp', 'CityOrigin', 'LatOrigin', 'LngOrigin', 'CityDestination', 'LatDestination', 'LngDestination', 'Distance', 'Source', 'Equip', 'StateOrigin', 'HubOrigin', 'StateDestination', 'HubDestination', 'RatePerMile', 'Weight', 'Size', 'Broker_Shipper']


(57573489, 20)

In [11]:
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        size_gb = obj.memory_usage(deep=True).sum() / (1024 ** 3)
        print(f"{name}: {obj.shape}, {size_gb:.2f} GB")

dfdrive: (14178352, 20), 3.61 GB
df123: (1555811, 19), 0.32 GB
dfaws: (54897563, 19), 12.29 GB
df_full: (70631726, 20), 17.18 GB
df_dupp: (62544534, 20), 15.06 GB
dfgenduplicates: (57573489, 20), 13.82 GB


In [12]:
del dfaws
del df_full
del df_dupp

In [13]:
dfgenduplicates['ID'] = dfgenduplicates['ID'].astype(str)
dfgenduplicates['Posted'] = pd.to_datetime(dfgenduplicates['Posted'],errors='coerce')
dfgenduplicates['PickUp'] = pd.to_datetime(dfgenduplicates['PickUp'],errors='coerce')
dfgenduplicates = dfgenduplicates.replace(['None', 'nan', 'NaN', ''], np.nan)
dfgenduplicates['Weight'] = dfgenduplicates['Weight'].astype(str)
dfgenduplicates['Size'] = dfgenduplicates['Size'].astype(str)

In [14]:
dfgenduplicates.to_parquet('parquetloadsfull.parquet')